# Chapter 4: Guidelines and Standard Metrics for Evaluating LLMs
**Module 04: Introduction to LLMs in Python**

> Source integrated from `chapter4.pdf`.

## Learning Objectives
- Evaluate classification outputs with accuracy, precision, recall, and F1.
- Use the Hugging Face `evaluate` library.
- Apply specialized language metrics: perplexity, ROUGE, BLEU, METEOR, and Exact Match.
- Explain human feedback, RLHF, reward models, toxicity, regard, and hallucination risks.

## Chapter Map
| Section | Focus |
|---|---|
| 4.1 | Classification accuracy and `evaluate` |
| 4.2 | Matching metrics to LLM tasks |
| 4.3 | Specialized language metrics |
| 4.4 | Human feedback and RLHF |
| 4.5 | Ethical challenges, hallucinations, toxicity, and regard |


## 4.1 Classification Accuracy

Accuracy measures the percentage of correctly predicted labels.

\[
Accuracy = rac{TP + TN}{TP + TN + FP + FN}
\]

The PDF evaluates a sentiment-analysis pipeline against four labeled examples.


In [ ]:
# If needed, install dependencies first:
# pip install transformers evaluate scikit-learn torch rouge_score sacrebleu nltk trl

from transformers import pipeline
from sklearn.metrics import accuracy_score

sentiment_analysis = pipeline("sentiment-analysis")

test_examples = [
    {"text": "I love this product!", "label": 1},
    {"text": "The service was terrible.", "label": 0},
    {"text": "This movie is amazing.", "label": 1},
    {"text": "I'm disappointed with the quality.", "label": 0},
]

predictions = sentiment_analysis([example["text"] for example in test_examples])
true_labels = [example["label"] for example in test_examples]
predicted_labels = [1 if pred["label"] == "POSITIVE" else 0 for pred in predictions]
accuracy = accuracy_score(true_labels, predicted_labels)

print("Test Examples:")
for example, pred_label in zip(test_examples, predicted_labels):
    print(f"Text: {example['text']}, Prediction: {pred_label}")
print(f"Accuracy: {accuracy:.2%}")


## 4.2 The `evaluate` Library

| Category | Purpose |
|---|---|
| Metric | Evaluate model performance against references or labels |
| Comparison | Compare two model outputs |
| Measurement | Inspect dataset or generated-text properties |

Metrics expose their required input schema through `.features`, commonly `predictions` and `references`.


In [ ]:
import evaluate

accuracy = evaluate.load("accuracy")
print(accuracy.description)

f1_description = evaluate.load("f1").description
print(f1_description)


In [ ]:
import evaluate

accuracy = evaluate.load("accuracy")
print(accuracy.features)

f1 = evaluate.load("f1")
print(f1.features)

pearson_corr = evaluate.load("pearsonr")
print(pearson_corr.features)


In [ ]:
import evaluate

accuracy = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall = evaluate.load("recall")
f1 = evaluate.load("f1")

real_labels = [0, 1, 0, 1, 1]
predicted_labels = [0, 0, 0, 1, 1]

print(accuracy.compute(references=real_labels, predictions=predicted_labels))
print(precision.compute(references=real_labels, predictions=predicted_labels))
print(recall.compute(references=real_labels, predictions=predicted_labels))
print(f1.compute(references=real_labels, predictions=predicted_labels))


## 4.3 LLM Tasks and Metrics

| Task | Useful Metrics | Notes |
|---|---|---|
| Classification | Accuracy, precision, recall, F1 | Account for class imbalance and error cost |
| Text generation | Perplexity, human evaluation | Lower perplexity does not guarantee better usefulness |
| Summarization | ROUGE, factuality review | ROUGE measures overlap, not truthfulness |
| Translation | BLEU, METEOR, human evaluation | BLEU rewards n-gram overlap; METEOR adds richer matching |
| Question answering | Exact Match, F1 | EM is strict; F1 gives partial credit |
| Safety and bias | Toxicity, regard, bias audits | Pair metrics with qualitative review |

> **Important:** Be comprehensive. Use multiple metrics and domain-specific KPIs whenever possible.


## 4.4 Perplexity in Text Generation

Perplexity measures a model's ability to predict the next word accurately and confidently. It is calculated from model likelihoods; lower is generally better.


In [ ]:
import evaluate
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

prompt = "Latest research findings in Antarctica show"
prompt_ids = tokenizer.encode(prompt, return_tensors="pt")
output = model.generate(prompt_ids, max_length=17)
generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
print(generated_text)

perplexity = evaluate.load("perplexity", module_type="metric")
results = perplexity.compute(model_id="gpt2", predictions=[generated_text])
print(results["mean_perplexity"])


## 4.5 ROUGE for Summarization

ROUGE evaluates overlap between generated summaries and reference summaries.

| Variant | Measures |
|---|---|
| `rouge1` | Unigram overlap |
| `rouge2` | Bigram overlap |
| `rougeL` | Longest common subsequence overlap |


In [ ]:
import evaluate

rouge = evaluate.load("rouge")
predictions = ["""as we learn more about the frequency and
        size distribution of exoplanets, we are discovering
        that terrestrial planets are exceedingly common."""]
references = ["""The more we learn about the frequency and
        size distribution of exoplanets, the more confident we
        are that they are exceedingly common."""]
results = rouge.compute(predictions=predictions, references=references)
print(results)


## 4.6 BLEU and METEOR for Translation

| Metric | Captures | Limitation |
|---|---|---|
| BLEU | N-gram precision against references | Can miss semantic similarity |
| METEOR | Precision, recall, stemming, synonyms, and word order | More computationally expensive |


In [ ]:
import evaluate
from transformers import pipeline

bleu = evaluate.load("bleu")
translator = pipeline("translation", model="Helsinki-NLP/opus-mt-es-en")

input_text = "Que hermoso dia"
references = [["What a gorgeous day", "What a beautiful day"]]
translated_outputs = translator(input_text)
translated_sentence = translated_outputs[0]["translation_text"]
print("Translation:", translated_sentence)

results = bleu.compute(predictions=[translated_sentence], references=references)
print(results)


In [ ]:
import evaluate

bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")

pred = ["""He thought it right and necessary to become a
        knight-errant, roaming the world in armor, seeking
        adventures and practicing the deeds he had read about
        in chivalric tales."""]
ref = [["""He believed it was proper and essential to transform
       into a knight-errant, traveling the world in armor,
       pursuing adventures, and enacting the heroic deeds he
       had encountered in tales of chivalry."""]]

results_bleu = bleu.compute(predictions=pred, references=ref)
results_meteor = meteor.compute(predictions=pred, references=[r[0] for r in ref])
print("Bleu:", results_bleu["bleu"])
print("Meteor:", results_meteor["meteor"])


## 4.7 Exact Match in Question Answering

Exact Match is 1 only when an answer exactly matches the reference. It is usually used with F1 to avoid losing partial-credit signal.


In [ ]:
import evaluate
from evaluate import load

em_metric = load("exact_match")
exact_match = evaluate.load("exact_match")

predictions = ["The cat sat on the mat.", "Theaters are great.", "It's like comparing oranges and apples."]
references = ["The cat sat on the mat?", "Theaters are great.", "It's like comparing apples and oranges."]
results = exact_match.compute(references=references, predictions=predictions)
print(results)


## 4.8 Human Feedback and RLHF

Objective metrics cannot fully capture subjective quality. Human feedback can guide optimization for truthfulness, helpfulness, originality, level of detail, concision, and context fit.

### RLHF Workflow
1. Start with an initial LLM.
2. Collect human preferences and train a reward model.
3. Optimize the LLM with reinforcement learning, often PPO.

| Component | Role |
|---|---|
| Reward model | Predicts a reward for prompt-response samples |
| Reference model | Anchors optimization to the original behavior |
| PPO trainer | Updates the policy model using reward feedback |


In [ ]:
# Optional RLHF setup example. Requires `trl` and can be resource intensive.
import torch
from transformers import AutoTokenizer
from trl import PPOTrainer, PPOConfig, create_reference_model, AutoModelForCausalLMWithValueHead
from trl.core import respond_to_batch

model = AutoModelForCausalLMWithValueHead.from_pretrained("gpt2")
model_ref = create_reference_model(model)
tokenizer = AutoTokenizer.from_pretrained("gpt2")

if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": "[PAD]"})

query_txt = "My plan today is to "
input_ids = tokenizer.encode(query_txt, return_tensors="pt")
response = respond_to_batch(model, input_ids)
ppo_config = PPOConfig(batch_size=1)
ppo_trainer = PPOTrainer(ppo_config, model, model_ref, tokenizer)
reward = [torch.tensor(1.0)]

# Uncomment to perform one PPO update.
# train_stats = ppo_trainer.step([input_ids[0]], [response[0]], reward)


## 4.9 Challenges and Ethical Considerations

| Challenge | Why It Matters |
|---|---|
| Multi-language support | Language diversity, data availability, and transferability vary widely |
| Open vs. closed LLMs | Transparency and accessibility must be balanced with responsible use |
| Scalability | Larger models increase compute, cost, and deployment complexity |
| Bias | Training data can encode unfair patterns |
| Hallucinations | Generated text can sound truthful while being false or nonsensical |

### Hallucination Reduction Strategies
- Use diverse and representative training data.
- Audit outputs for bias and apply bias removal techniques.
- Fine-tune for specific use cases in sensitive applications.
- Use careful prompt engineering and evaluation.


## 4.10 Toxicity

Toxicity measures harmful or abusive language with a pretrained classifier. `aggregation="maximum"` returns the maximum toxicity score across inputs.


In [ ]:
from evaluate import load

toxicity_metric = load("toxicity")
texts_1 = ["Everyone in the team adores him", "He is a true genius, pure talent"]
texts_2 = ["Nobody in the team likes him", "He is a useless 'good-for-nothing'"]

toxicity_results = toxicity_metric.compute(predictions=texts_1, aggregation="maximum")
print("Toxicity Sentences 1:", toxicity_results)

toxicity_results = toxicity_metric.compute(predictions=texts_2, aggregation="maximum")
print("Toxicity Sentences 2:", toxicity_results)


## 4.11 Regard

Regard measures language polarity and biased perception toward groups. Evaluate groups separately and compare score distributions.


In [ ]:
from evaluate import load

regard = load("regard")
group1 = ["abc are described as loyal employees", "abc are ambitious in their career expectations"]
group2 = ["abc are known for causing lots of team conflicts", "abc are verbally violent"]

polarity_results_1 = regard.compute(data=group1)
for result in polarity_results_1["regard"]:
    print(result)

polarity_results_2 = regard.compute(data=group2)
for result in polarity_results_2["regard"]:
    print(result)


## Course Wrap-Up

| Chapter | Core Skill |
|---|---|
| Chapter 1 | Understand LLMs and run pretrained pipelines |
| Chapter 2 | Build transformer components with attention, masks, encoders, and decoders |
| Chapter 3 | Use pretrained models for classification, generation, summarization, translation, QA, and fine-tuning |
| Chapter 4 | Evaluate LLMs with task metrics, human feedback, and safety-aware measurements |

## Chapter Summary
- Choose metrics based on task behavior, not convenience.
- Combine automated metrics with human and domain evaluation.
- RLHF uses preference data to optimize behavior beyond supervised labels.
- Bias, toxicity, and hallucination checks are part of responsible LLM evaluation.
